In [1]:
# ============================================================================
# MNA-net replication, Mamba fusion substitution, and pure Vision Mamba
# ----------------------------------------------------------------------------
# 3 experiments on the same MNI-space, SynthStripped data:
#
#   EXP 1  Replicate Vo et al.'s MNA-net stage 3 (patch fusion by
#           concatenation + dense layer), using the frozen stage-1
#           patch encoders and stage-2 modality attention.
#           Published result: 82.9% / 85.7% / 80.0%
#
#   EXP 2  Mamba replaces the concatenation -- the 27 patches become a sequence with positional
#           embeddings rather than a flat 5400-dim vector.
#
#   EXP3 3  Pure Vision Mamba on the same volumes: no CNN, no pretrained
#           weights, just patch tokenisation straight into Mamba.
#           MRI, PET and multimodal.
#
# Cohort: 209 subjects (OAS30065 excluded -- corrupt source PET)
# ============================================================================

import os, glob, time, random, json, importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
# ----------------------------------------------------------------------------
# All three parts read from mnanet_format/, which holds the MNI-registered,
# BET+SynthStripped data
# ----------------------------------------------------------------------------

BASE     = 'D:/mamba_model/mnanet_format'
JAMIE    = f'{BASE}/Multimodal-Attention-based-Neural-Networks-for-the-Prediction-of-Cognitive-Decline-main'
RESNET   = f'{JAMIE}/Train Models/ResNetV2.py'
CKPT_DIR = f'{BASE}/patch_temp'                  # frozen encoders
CACHE    = f'{BASE}/3_patch_cache'               # 27-patch tensors + features
WB_CACHE = f'{BASE}/4_volume_cache_mni_wb'       # whole MNI volumes for exp 3

OUT_CKPT = 'D:/mamba_model/checkpoints_mnanet_final'
RESULTS  = 'D:/mamba_model/mnanet_final_results.json'
os.makedirs(OUT_CKPT, exist_ok=True)

SEEDS = [123]       
PREFIX = 'PATCH_TEMP'           # encoder checkpoints

VO_PUBLISHED = dict(acc=0.829, tpr=0.857, tnr=0.800)

for p, label in [(RESNET, 'ResNetV2.py'),
                 (f'{CACHE}/mm_features_ss.pt', 'SynthStrip features'),
                 (f'{CACHE}/multimodal_labels.pt', 'labels'),
                 (WB_CACHE, 'MNI volume cache')]:
    print(f"  {'OK ' if os.path.exists(p) else 'MISSING'}  {label}")

results = {}

  OK   ResNetV2.py
  OK   SynthStrip features
  OK   labels
  OK   MNI volume cache


In [3]:
F_ss = torch.load(f'{CACHE}/mm_features_ss.pt', weights_only=False)
lab  = torch.load(f'{CACHE}/multimodal_labels.pt', weights_only=False)

y_tr = lab['train_y'].float()
y_va = lab['val_y'].float()
y_te = lab['test_y'].float()

for k in ['tr_mri', 'tr_pet', 'va_mri', 'va_pet', 'te_mri', 'te_pet']:
    print(f"  {k}: {tuple(F_ss[k].shape)}")
print(f"\nlabels: train {len(y_tr)} | val {len(y_va)} | test {len(y_te)}")
print(f"test positives: {int(y_te.sum())} of {len(y_te)}")

  tr_mri: (500, 27, 100)
  tr_pet: (500, 27, 100)
  va_mri: (42, 27, 100)
  va_pet: (42, 27, 100)
  te_mri: (42, 27, 100)
  te_pet: (42, 27, 100)

labels: train 500 | val 42 | test 42
test positives: 22 of 42


In [4]:
class MNAAttention(nn.Module):
    """concatenate 27 fused patches, one dense layer."""
    def __init__(self, d_model=100, n_heads=4, dropout=0.4):
        super().__init__()
        self.att  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(d_model * 2 * 27, 1)      # 5400 -> 1

    def forward(self, mri, pet):
        B = mri.shape[0]
        fused = []
        for j in range(27):
            x = torch.stack([self.drop(mri[:, j]), self.drop(pet[:, j])])   # (2,B,100)
            x, _ = self.att(x, x, x)
            fused.append(x.permute(1, 0, 2).reshape(B, 200))
        return torch.sigmoid(self.fc(self.drop(torch.cat(fused, dim=1))))


class MNAMamba(nn.Module):
    """Mamba over the 27 patches instead of concatenation."""
    def __init__(self, d_model=100, n_heads=4, n_layers=2, d_state=16, dropout=0.4):
        super().__init__()
        self.att  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        self.proj = nn.Linear(d_model * 2, d_model)      # 200 -> 100 per patch
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder    = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
        self.pos        = nn.Embedding(27, d_model)      # which patch is where
        with torch.no_grad():
            self.pos.weight.mul_(0.02)
        self.drop       = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, mri, pet):
        B = mri.shape[0]
        fused = []
        for j in range(27):
            x = torch.stack([self.drop(mri[:, j]), self.drop(pet[:, j])])
            x, _ = self.att(x, x, x)
            fused.append(self.proj(x.permute(1, 0, 2).reshape(B, 200)))
        t = torch.stack(fused, dim=1) + self.pos.weight[None]      # (B,27,100)
        t = self.final_norm(self.encoder(t))
        return torch.sigmoid(self.classifier(self.drop(t.mean(1))))


for cls in (MNAAttention, MNAMamba):
    m = cls().to(device)
    with torch.no_grad():
        o = m(F_ss['tr_mri'][:4].to(device), F_ss['tr_pet'][:4].to(device))
    print(f"{cls.__name__:14s} out {tuple(o.shape)} "
          f"range {o.min():.3f}-{o.max():.3f}  "
          f"params {sum(p.numel() for p in m.parameters()):,}")
    del m
torch.cuda.empty_cache()

MNAAttention   out (4, 1) range 0.484-0.525  params 45,801
MNAMamba       out (4, 1) range 0.360-0.479  params 238,901


In [5]:
# ----------------------------------------------------------------------------
# Hyperparameters follow MNA-net: SGD with momentum 0.9,
# lr 1e-4, BCE loss, batch size 10, early stopping on validation loss.
# ----------------------------------------------------------------------------
def run_fusion(seed, model_cls, features, lr=1e-4, patience=25,
               max_epochs=500, log_every=10):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)

    m    = model_cls().to(device)
    opt  = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9)
    crit = nn.BCELoss()
    tl   = DataLoader(TensorDataset(features['tr_mri'], features['tr_pet'], y_tr),
                      batch_size=10, shuffle=True)

    Xv = (features['va_mri'].to(device), features['va_pet'].to(device))
    Xt = (features['te_mri'].to(device), features['te_pet'].to(device))
    yv, yt = y_va.to(device), y_te.to(device)

    best, no_imp, state, best_ep = np.inf, 0, None, 0
    t0 = time.time()

    print(f"\n--- {model_cls.__name__} seed {seed} ---")
    print(f"{'ep':>5} | {'train':>8} | {'val':>8} | {'acc':>6} | {'tpr':>6} | {'tnr':>6}")
    print('-' * 56)

    for ep in range(max_epochs):
        m.train(); tot = 0
        for a, b, lb in tl:
            opt.zero_grad()
            loss = crit(m(a.to(device), b.to(device)).flatten(), lb.to(device))
            loss.backward(); opt.step(); tot += loss.item()
        trl = tot / len(tl)

        m.eval()
        with torch.no_grad():
            vp = m(*Xv).flatten()
            vl = crit(vp, yv).item()
            vpred = (vp >= 0.5).float().cpu().numpy()
        vy = yv.cpu().numpy()
        vacc = float((vpred == vy).mean())
        vtpr = float(recall_score(vy, vpred, zero_division=0))
        vtnr = float(recall_score(vy, vpred, pos_label=0, zero_division=0))

        if ep % log_every == 0 or ep == 0:
            print(f"{ep:>5} | {trl:>8.4f} | {vl:>8.4f} | "
                  f"{vacc:>6.3f} | {vtpr:>6.3f} | {vtnr:>6.3f}")

        if vl < best:
            best, no_imp, best_ep = vl, 0, ep
            state = {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            no_imp += 1
            if no_imp > patience:
                print(f"  early stop at {ep}, best epoch {best_ep}")
                break

    m.load_state_dict(state); m.eval()
    with torch.no_grad():
        probs = m(*Xt).flatten().cpu().numpy()
    preds, yy = (probs >= 0.5).astype(int), yt.cpu().numpy()

    acc = float((preds == yy).mean())
    tpr = float(recall_score(yy, preds, zero_division=0))
    tnr = float(recall_score(yy, preds, pos_label=0, zero_division=0))
    npar = sum(p.numel() for p in m.parameters())
    dt = time.time() - t0

    print(f"  >>> TEST  Acc {acc*100:.1f}%  TPR {tpr*100:.1f}%  TNR {tnr*100:.1f}%  "
          f"| best_ep {best_ep} | {dt/60:.1f} min")
    del m; torch.cuda.empty_cache()

    return dict(seed=seed, acc=acc, tpr=tpr, tnr=tnr, best_epoch=best_ep,
                train_time_sec=dt, n_params=npar)

In [6]:
# EXP 1
print("=== PART 1: MNA-net replication ===")
print(f"Vo published: {VO_PUBLISHED['acc']*100:.1f}% / "
      f"{VO_PUBLISHED['tpr']*100:.1f}% / {VO_PUBLISHED['tnr']*100:.1f}%")
results['mnanet_attention'] = [run_fusion(s, MNAAttention, F_ss) for s in SEEDS]

=== PART 1: MNA-net replication ===
Vo published: 82.9% / 85.7% / 80.0%

--- MNAAttention seed 123 ---
   ep |    train |      val |    acc |    tpr |    tnr
--------------------------------------------------------
    0 |   0.6899 |   0.6727 |  0.595 |  1.000 |  0.190
   10 |   0.5351 |   0.5455 |  0.738 |  0.762 |  0.714
   20 |   0.4793 |   0.5091 |  0.738 |  0.762 |  0.714
   30 |   0.4532 |   0.4968 |  0.738 |  0.762 |  0.714
   40 |   0.4324 |   0.4931 |  0.762 |  0.810 |  0.714
   50 |   0.4321 |   0.4909 |  0.762 |  0.810 |  0.714
   60 |   0.4282 |   0.4890 |  0.762 |  0.810 |  0.714
   70 |   0.4022 |   0.4892 |  0.786 |  0.857 |  0.714
   80 |   0.4089 |   0.4886 |  0.786 |  0.857 |  0.714
   90 |   0.4106 |   0.4859 |  0.786 |  0.857 |  0.714
  100 |   0.4068 |   0.4867 |  0.786 |  0.857 |  0.714
  110 |   0.4043 |   0.4823 |  0.786 |  0.857 |  0.714
  120 |   0.3950 |   0.4862 |  0.786 |  0.857 |  0.714
  130 |   0.3952 |   0.4814 |  0.786 |  0.857 |  0.714
  140 |   0.398

In [7]:
# EXP 2
print("=== PART 2: Mamba substituted into fusion ===")
print("Same frozen encoders, same modality attention, same features.")
results['mnanet_mamba'] = [run_fusion(s, MNAMamba, F_ss) for s in SEEDS]

=== PART 2: Mamba substituted into fusion ===
Same frozen encoders, same modality attention, same features.

--- MNAMamba seed 123 ---
   ep |    train |      val |    acc |    tpr |    tnr
--------------------------------------------------------
    0 |   0.6937 |   0.6900 |  0.571 |  0.333 |  0.810
   10 |   0.6775 |   0.6536 |  0.667 |  0.381 |  0.952
   20 |   0.6437 |   0.6066 |  0.738 |  0.667 |  0.810
   30 |   0.6133 |   0.5630 |  0.714 |  0.714 |  0.714
   40 |   0.5750 |   0.5234 |  0.738 |  0.667 |  0.810
   50 |   0.5368 |   0.5092 |  0.714 |  0.714 |  0.714
   60 |   0.5048 |   0.5179 |  0.714 |  0.667 |  0.762
   70 |   0.5140 |   0.5341 |  0.714 |  0.714 |  0.714
  early stop at 75, best epoch 49
  >>> TEST  Acc 76.2%  TPR 59.1%  TNR 95.0%  | best_ep 49 | 2.3 min


In [8]:
# Compare EXP 1 and EXP 2
def line(rs, name):
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    sd = lambda v: np.std(v, ddof=1) * 100 if len(v) > 1 else 0.0
    print(f"{name:34s} {np.mean(a)*100:5.1f}±{sd(a):4.1f}%  "
          f"{np.mean(t)*100:5.1f}%  {np.mean(n)*100:5.1f}%   "
          f"{rs[0]['n_params']:>9,}p")

print(f"{'':34s} {'Acc':>11s}  {'TPR':>6s}  {'TNR':>6s}   {'params':>10s}")
print(f"{'MNA-net (Vo et al., published)':34s} "
      f"{VO_PUBLISHED['acc']*100:10.1f}%  {VO_PUBLISHED['tpr']*100:5.1f}%  "
      f"{VO_PUBLISHED['tnr']*100:5.1f}%")
line(results['mnanet_attention'], 'PART 1: replication (attention)')
line(results['mnanet_mamba'],     'PART 2: Mamba fusion')

                                           Acc     TPR     TNR       params
MNA-net (Vo et al., published)           82.9%   85.7%   80.0%
PART 1: replication (attention)     83.3± 0.0%   81.8%   85.0%      45,801p
PART 2: Mamba fusion                76.2± 0.0%   59.1%   95.0%     238,901p


In [9]:
# EXP 3
# mamba using MNA net data
# 88x108x88 crop -> 11x13x11 grid -> 1,573 tokens per volume.

lab_wb = torch.load(f'{WB_CACHE}/mni_wb_labels.pt', weights_only=False)
VOL = {}
for t in ['tr_mri', 'va_mri', 'te_mri']:
    VOL[t] = torch.load(f'{WB_CACHE}/mni_wb_vol_{t}.pt', weights_only=False)
for t in ['tr_pet', 'va_pet', 'te_pet']:
    VOL[t] = torch.load(f'{WB_CACHE}/mni_wb_ss_vol_{t}.pt', weights_only=False)   # SynthStrip

y_tr_wb = lab_wb['train_y'].long()
y_va_wb = lab_wb['val_y'].long()
y_te_wb = lab_wb['test_y'].long()

for k, v in VOL.items():
    print(f"  {k}: {tuple(v.shape)}")
print(f"\nlabels: {len(y_tr_wb)} / {len(y_va_wb)} / {len(y_te_wb)}  "
      f"| test pos {int(y_te_wb.sum())}")

BRAIN_SHAPE = tuple(VOL['tr_mri'].shape[-3:])
BATCH = 4

def uni_loaders(mod):
    return (DataLoader(TensorDataset(VOL[f'tr_{mod}'], y_tr_wb), batch_size=BATCH, shuffle=True),
            DataLoader(TensorDataset(VOL[f'va_{mod}'], y_va_wb), batch_size=BATCH),
            DataLoader(TensorDataset(VOL[f'te_{mod}'], y_te_wb), batch_size=BATCH))

def mm_loaders():
    return (DataLoader(TensorDataset(VOL['tr_mri'], VOL['tr_pet'], y_tr_wb), batch_size=BATCH, shuffle=True),
            DataLoader(TensorDataset(VOL['va_mri'], VOL['va_pet'], y_va_wb), batch_size=BATCH),
            DataLoader(TensorDataset(VOL['te_mri'], VOL['te_pet'], y_te_wb), batch_size=BATCH))

  tr_mri: (500, 1, 88, 108, 88)
  va_mri: (42, 1, 88, 108, 88)
  te_mri: (42, 1, 88, 108, 88)
  tr_pet: (500, 1, 88, 108, 88)
  va_pet: (42, 1, 88, 108, 88)
  te_pet: (42, 1, 88, 108, 88)

labels: 500 / 42 / 42  | test pos 22


In [10]:
class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))


class WBPatchEmbed3D(nn.Module):
    def __init__(self, brain_shape=BRAIN_SHAPE, patch_size=8, d_model=32):
        super().__init__()
        self.patch_size = patch_size
        self.gd, self.gh, self.gw = [s // patch_size for s in brain_shape]
        self.n_tokens = self.gd * self.gh * self.gw
        self.patch_conv   = nn.Conv3d(1, d_model, patch_size, stride=patch_size)
        self.depth_embed  = nn.Embedding(self.gd, d_model)
        self.height_embed = nn.Embedding(self.gh, d_model)
        self.width_embed  = nn.Embedding(self.gw, d_model)
        with torch.no_grad():
            for e in [self.depth_embed, self.height_embed, self.width_embed]:
                e.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.gd), torch.arange(self.gh),
                                 torch.arange(self.gw), indexing='ij')
        self.register_buffer('coordinates', torch.stack([d, h, w], -1).reshape(-1, 3),
                             persistent=False)

    def forward(self, volume):
        tokens = self.patch_conv(volume).flatten(2).transpose(1, 2)
        c = self.coordinates
        tokens = tokens + (self.depth_embed(c[:, 0]) + self.height_embed(c[:, 1])
                           + self.width_embed(c[:, 2]))[None]
        occ = F.max_pool3d((volume.abs() > 1e-6).float(),
                           kernel_size=self.patch_size, stride=self.patch_size)
        valid = occ.flatten(1).bool()
        return tokens * valid.unsqueeze(-1).to(tokens.dtype), valid


class WBBranch(nn.Module):
    def __init__(self, brain_shape=BRAIN_SHAPE, patch_size=8, d_model=32, n_layers=2):
        super().__init__()
        self.patch_embed = WBPatchEmbed3D(brain_shape, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers)
    def forward(self, volume):
        tokens, valid = self.patch_embed(volume)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        return (tokens * w).sum(1) / w.sum(1).clamp_min(1.0)


class PureVimModel(nn.Module):
    def __init__(self, d_model=32, n_layers=2, n_classes=2, dropout=0.4):
        super().__init__()
        self.branch = WBBranch(BRAIN_SHAPE, 8, d_model, n_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)
    def forward(self, v):
        return self.classifier(self.dropout(self.branch(v)))


class PureVimMultimodal(nn.Module):
    def __init__(self, d_model=32, n_layers=2, n_classes=2, dropout=0.4):
        super().__init__()
        self.mri_branch = WBBranch(BRAIN_SHAPE, 8, d_model, n_layers)
        self.pet_branch = WBBranch(BRAIN_SHAPE, 8, d_model, n_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)
    def forward(self, mri, pet):
        f = torch.cat([self.mri_branch(mri), self.pet_branch(pet)], dim=1)
        return self.classifier(self.dropout(f))


_m = PureVimModel()
print(f"tokens per volume: {_m.branch.patch_embed.n_tokens:,} "
      f"(grid {_m.branch.patch_embed.gd}x{_m.branch.patch_embed.gh}x{_m.branch.patch_embed.gw})")
print(f"params: {sum(p.numel() for p in _m.parameters()):,}")
del _m

tokens per volume: 1,573 (grid 11x13x11)
params: 45,122


In [11]:
# ----------------------------------------------------------------------------
# AdamW rather than SGD here, matching the rest of the Vision Mamba work.
# min_epochs guards against early stopping firing
# ----------------------------------------------------------------------------

def run_pure(seed, model_cls, loaders, mm, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-3, log_every=5):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    m    = model_cls(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt  = optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-3)
    sch  = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    def ev(loader):
        m.eval(); tot, P, L = 0, [], []
        with torch.no_grad():
            for batch in loader:
                if mm:
                    a, b, lb = [x.to(device) for x in batch]; out = m(a, b)
                else:
                    a, lb = [x.to(device) for x in batch];    out = m(a)
                tot += crit(out, lb).item()
                P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.cpu().numpy())
        return (tot/len(loader), float(np.mean(np.array(P) == np.array(L))),
                float(recall_score(L, P, zero_division=0)),
                float(specificity_score(L, P)))

    best, no_imp, best_ep, total = np.inf, 0, 0, 0
    path = f"{OUT_CKPT}/{prefix}_seed{seed}.pt"

    print(f"\n--- {prefix} seed {seed} ---")
    print(f"{'ep':>5} | {'train':>8} | {'val':>8} | {'acc':>6} | {'tpr':>6} | "
          f"{'tnr':>6} | {'sec':>5}")
    print('-' * 64)

    for ep in range(1, max_epochs):
        t0 = time.time(); m.train(); tot = 0
        for batch in tr:
            opt.zero_grad()
            if mm:
                a, b, lb = [x.to(device) for x in batch]; out = m(a, b)
            else:
                a, lb = [x.to(device) for x in batch];    out = m(a)
            loss = crit(out, lb); loss.backward(); opt.step(); tot += loss.item()
        trl = tot / len(tr)

        vl, vacc, vtpr, vtnr = ev(va)
        sch.step(vl)
        dt = time.time() - t0; total += dt

        if ep % log_every == 0 or ep == 1:
            print(f"{ep:>5} | {trl:>8.4f} | {vl:>8.4f} | {vacc:>6.3f} | "
                  f"{vtpr:>6.3f} | {vtnr:>6.3f} | {dt:>5.1f}")

        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(m.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop at {ep}, best epoch {best_ep}")
                break

    m.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = ev(te)
    npar = sum(p.numel() for p in m.parameters() if p.requires_grad)

    warn = '  *** best_ep < 5, may not have trained ***' if best_ep < 5 else ''
    print(f"  >>> TEST  Acc {acc*100:.1f}%  TPR {tpr*100:.1f}%  TNR {tnr*100:.1f}%  "
          f"| best_ep {best_ep} | {total/60:.1f} min{warn}")
    del m; torch.cuda.empty_cache()

    return dict(seed=seed, acc=acc, tpr=tpr, tnr=tnr, best_epoch=best_ep,
                train_time_sec=total, n_params=npar)

In [12]:
# EXP 3
print("=== EXP 3: Vision Mamba, MNI + SynthStrip ===")
print(f"{(BRAIN_SHAPE[0]//8)*(BRAIN_SHAPE[1]//8)*(BRAIN_SHAPE[2]//8):,} tokens per volume\n")

print("### MRI-only ###")
results['pure_mri'] = [run_pure(s, PureVimModel, uni_loaders('mri'), False,
                                'pure_mri') for s in SEEDS]

print("\n### PET-only ###")
results['pure_pet'] = [run_pure(s, PureVimModel, uni_loaders('pet'), False,
                                'pure_pet') for s in SEEDS]

print("\n### Multimodal ###")
results['pure_mm'] = [run_pure(s, PureVimMultimodal, mm_loaders(), True,
                               'pure_mm') for s in SEEDS]

=== EXP 3: Vision Mamba, MNI + SynthStrip ===
1,573 tokens per volume

### MRI-only ###

--- pure_mri seed 123 ---
   ep |    train |      val |    acc |    tpr |    tnr |   sec
----------------------------------------------------------------
    1 |   0.7189 |   0.7048 |  0.500 |  1.000 |  0.000 |   5.0
    5 |   0.7004 |   0.6949 |  0.500 |  0.000 |  1.000 |   4.2
   10 |   0.6950 |   0.6931 |  0.500 |  1.000 |  0.000 |   4.1
   15 |   0.6970 |   0.6940 |  0.500 |  1.000 |  0.000 |   4.1
   20 |   0.6933 |   0.6926 |  0.500 |  1.000 |  0.000 |   4.1
   25 |   0.6950 |   0.6924 |  0.500 |  1.000 |  0.000 |   4.1
   30 |   0.6919 |   0.6922 |  0.500 |  0.952 |  0.048 |   4.1
   35 |   0.6896 |   0.6912 |  0.524 |  0.810 |  0.238 |   4.1
   40 |   0.6835 |   0.6955 |  0.500 |  1.000 |  0.000 |   4.1
   45 |   0.6769 |   0.6898 |  0.548 |  0.429 |  0.667 |   4.1
   50 |   0.6375 |   0.7235 |  0.500 |  0.000 |  1.000 |   4.1
   55 |   0.5775 |   0.7155 |  0.571 |  0.714 |  0.429 |   4.1
 

In [13]:
# Results

def line(rs, name):
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]
    sd = lambda v: np.std(v, ddof=1) * 100 if len(v) > 1 else 0.0
    print(f"{name:36s} {np.mean(a)*100:5.1f}±{sd(a):4.1f}%  "
          f"{np.mean(t)*100:5.1f}%  {np.mean(n)*100:5.1f}%  "
          f"{rs[0]['n_params']:>9,}p  {np.mean(tm)/60:5.1f}m")

print(f"seeds: {SEEDS}\n")
print(f"{'':36s} {'Acc':>11s}  {'TPR':>6s}  {'TNR':>6s}  {'params':>10s}  {'train':>6s}")
print('-' * 84)
print(f"{'MNA-net (Vo et al., published)':36s} {VO_PUBLISHED['acc']*100:10.1f}%  "
      f"{VO_PUBLISHED['tpr']*100:5.1f}%  {VO_PUBLISHED['tnr']*100:5.1f}%")
print()
line(results['mnanet_attention'], 'EXP 1  replication (attention)')
line(results['mnanet_mamba'],     'EXP 2  Mamba fusion')
print()
line(results['pure_mri'], 'EXP 3  pure Vim, MRI')
line(results['pure_pet'], 'EXP 3  pure Vim, PET')
line(results['pure_mm'],  'EXP 3  pure Vim, multimodal')

with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS}")

seeds: [123]

                                             Acc     TPR     TNR      params   train
------------------------------------------------------------------------------------
MNA-net (Vo et al., published)             82.9%   85.7%   80.0%

EXP 1  replication (attention)        83.3± 0.0%   81.8%   85.0%     45,801p    6.0m
EXP 2  Mamba fusion                   76.2± 0.0%   59.1%   95.0%    238,901p    2.3m

EXP 3  pure Vim, MRI                  59.5± 0.0%   50.0%   70.0%     45,122p    4.1m
EXP 3  pure Vim, PET                  64.3± 0.0%   45.5%   85.0%     45,122p    2.9m
EXP 3  pure Vim, multimodal           57.1± 0.0%   50.0%   65.0%     90,242p    4.8m

saved D:/mamba_model/mnanet_final_results.json
